# 【講師用】Ch.3 画像処理入門（Pillow × NumPy）完全解説

> 🔒 **このノートブックは講師専用です。受講者には配布しないでください。**

---

## 📢 講師向け冒頭ガイド

### このチャプターで最も伝えたいこと
- **「画像 = 数値の2次元配列」** この1点がこのチャプターのすべて
- 画像も数値だから、pandas や numpy で扱えるし、機械学習の入力にもなる
- 実務での画像処理（製造業の外観検査など）への入口として体験させる

### よくある受講者の躓きポイント
| 躓き | 対処法 |
|------|--------|
| `Image.fromarray` の引数がわからない | 「mode='L' がグレースケール。とりあえず覚えてOK」と伝える |
| `/16.0 * 255` のスケール変換の意味がわからない | 「digits の値は0〜16、Pillow は0〜255を期待する。単位変換と同じ」と説明 |
| `np.array(img)` と `img_array` が別物になる意味がわからない | 「PIL ↔ numpy の変換は行き来できる。加工は PIL、計算は numpy」と整理 |

### 時間配分目安（座学20分）
- 3.1 画像データの正体：6分
- 3.2 Pillowで画像を操作する：6分
- 3.3 フィルタ処理：4分
- 3.4 ピクセル値の統計：4分

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageFilter
from sklearn.datasets import load_digits
import japanize_matplotlib

# digits データセットの読み込み
digits = load_digits()

print('画像データの形状:', digits.images.shape)
print('  → {}枚の画像、各画像は{}×{}ピクセル'.format(*digits.images.shape))
print('ラベル（数字）の種類:', np.unique(digits.target))

### 📢 digits データセットの説明スクリプト
```
「1797枚、各画像は8×8ピクセルです。
  今日のスマートフォンのカメラは数千万画素ですが、
  この画像は 8×8=64 ピクセルという非常に粗いデータです。
  それでも AI は数字を認識できます。それがこのチャプターで体感することです。」
```

---
## 3.1 画像データの正体：数値の配列

### 📢 講師メモ
ここが最重要セクションです。ピクセルの数値配列をそのまま画面に出力して見せることで
「画像 = 数値」という概念を体感してもらいます。

In [ ]:
# 1枚目の画像（数字「0」）を数値の配列として表示
img_array = digits.images[0]

print('=== 数値配列（8×8）===')
print(img_array.astype(int))
print()
print('ラベル（正解の数字）:', digits.target[0])
print('値の範囲: 0（黒）〜 16（白）')

### 📢 数値配列の解説スクリプト
```
「この数字の羅列が画像です。
  0 に近いほど暗い（黒）、16 に近いほど明るい（白）です。
  数字「0」の形を想像してみてください。
  中心が空洞（値が低い）で、周囲に輪郭（値が高い）がある形になっていますね。」

（受講者に質問）
「この配列を見て、これが何の数字か当てられますか？」
```

In [ ]:
# 数値配列と画像を並べて表示
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# 左：数値配列をテキストで表示
axes[0].axis('off')
table_text = ''
for row in img_array.astype(int):
    table_text += '  '.join(f'{v:2d}' for v in row) + '\n'
axes[0].text(0.05, 0.95, table_text, transform=axes[0].transAxes,
             fontsize=14, va='top', family='monospace')
axes[0].set_title('数値配列（0〜16）')

# 右：画像として表示
axes[1].imshow(img_array, cmap='gray_r')  # gray_r: 白黒反転
axes[1].set_title(f'画像として表示（ラベル: {digits.target[0]}）')
axes[1].axis('off')

plt.suptitle('「画像 = 数値の配列」の証明', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# 手書き数字を10枚並べて表示
fig, axes = plt.subplots(2, 5, figsize=(12, 5))

for i, ax in enumerate(axes.flat):
    ax.imshow(digits.images[i], cmap='gray_r')
    ax.set_title(f'ラベル: {digits.target[i]}', fontsize=12)
    ax.axis('off')

plt.suptitle('手書き数字データセット（最初の10枚）', fontsize=14)
plt.tight_layout()
plt.show()

---
## 3.2 Pillow（PIL）で画像を操作する

### 📢 講師メモ
Pillow は「画像を操作する道具箱」と説明します。
numpy array との変換を繰り返し強調してください：
- 画像の加工 → Pillow
- 数値の計算・統計 → numpy
- この2つを行き来するのが基本パターン

In [ ]:
# ─── numpy array → PIL Image への変換 ───────────────────────────
#
# なぜスケール変換が必要か？
# digits の値：0〜16 の整数
# Pillow の期待値：0〜255 の整数（uint8）
# → 16.0 で割って 255 をかける（比例配分）

img_array_scaled = (digits.images[0] / 16.0 * 255).astype(np.uint8)

print('変換前の値の範囲: 0 〜', int(digits.images[0].max()))
print('変換後の値の範囲: 0 〜', int(img_array_scaled.max()))

# mode='L' = グレースケール（Luminance の L）
img_pil = Image.fromarray(img_array_scaled, mode='L')

print('PIL Image サイズ:', img_pil.size)  # (幅, 高さ) の順
print('PIL Image モード:', img_pil.mode)   # L = グレースケール

### 📢 よく聞かれる質問と回答

**Q: `mode='L'` の意味は？**
> A: L は Luminance（輝度）の略。グレースケール画像を表す。
> カラーなら `mode='RGB'`（赤・緑・青の3チャンネル）を使う。

**Q: `uint8` とは？**
> A: Unsigned Integer 8-bit の略。0〜255 の範囲の整数。
> 画像のピクセル値は伝統的に 0〜255 で表現されるため、この型を使う。

In [ ]:
# ─── リサイズ：小さい画像を大きくする ────────────────────────────
# 8×8 → 80×80（10倍に拡大）
# Image.NEAREST = 最近傍補間（ぼかさずに拡大。ドット絵のような見た目になる）

img_large = img_pil.resize((80, 80), Image.NEAREST)

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(img_pil, cmap='gray_r')
axes[0].set_title(f'元画像: {img_pil.size}\n（8×8ピクセル）')
axes[0].axis('off')

axes[1].imshow(img_large, cmap='gray_r')
axes[1].set_title(f'リサイズ後: {img_large.size}\n（80×80ピクセル）')
axes[1].axis('off')

plt.suptitle('リサイズ（8×8 → 80×80）', fontsize=13)
plt.tight_layout()
plt.show()

### 📢 解説ポイント
- 元の 8×8 はモザイクのようだが、80×80 にすると「0」の形がわかりやすくなる
- `Image.NEAREST`（最近傍補間）はピクセルをそのまま拡大する → ドット絵のような見た目
- 実務では `Image.LANCZOS`（高品質な補間）を使うことが多いが、今回の目的には NEAREST で十分

---
## 3.3 フィルタ処理：画像を変換する

### 📢 講師メモ
フィルタ処理の「仕組み（畳み込み）」は深入りしなくて大丈夫です。
「周辺ピクセルの値を使って計算する」というイメージだけ伝えれば十分です。
「こんなに簡単に画像を変換できる」という体験に集中させましょう。

In [ ]:
# 作業用関数：指定した数字の画像を 80×80 PIL Image として返す
def prepare_digit_image(digit_idx):
    """指定インデックスの digits 画像を 80×80 の PIL Image として返す"""
    arr = (digits.images[digit_idx] / 16.0 * 255).astype(np.uint8)
    return Image.fromarray(arr, mode='L').resize((80, 80), Image.NEAREST)

# 数字「8」で試す（最初の「8」のインデックスを取得）
idx_8 = np.where(digits.target == 8)[0][0]
img_base = prepare_digit_image(idx_8)

# 各フィルタを適用
img_blur    = img_base.filter(ImageFilter.BLUR)       # ぼかし
img_sharp   = img_base.filter(ImageFilter.SHARPEN)    # シャープ化
img_contour = img_base.filter(ImageFilter.CONTOUR)    # 輪郭抽出
img_emboss  = img_base.filter(ImageFilter.EMBOSS)     # 浮き彫り

# 5つ並べて比較
fig, axes = plt.subplots(1, 5, figsize=(15, 3))
images = [img_base, img_blur, img_sharp, img_contour, img_emboss]
titles = ['元画像', 'BLUR\n（ぼかし）', 'SHARPEN\n（シャープ）', 'CONTOUR\n（輪郭）', 'EMBOSS\n（浮き彫り）']

for ax, img, title in zip(axes, images, titles):
    ax.imshow(img, cmap='gray_r')
    ax.set_title(title, fontsize=10)
    ax.axis('off')

plt.suptitle(f'フィルタ処理の比較（数字: {digits.target[idx_8]}）', fontsize=13)
plt.tight_layout()
plt.show()

### 📢 各フィルタの解説スクリプト

```
BLUR（ぼかし）:
「周辺ピクセルの平均をとります。結果、エッジがなめらかになります。
  ノイズ除去によく使われます。」

SHARPEN（シャープ化）:
「中心ピクセルを強調し、周辺との差を際立たせます。
  エッジが強調されて「くっきり」した印象になります。」

CONTOUR（輪郭抽出）:
「隣接ピクセルとの差分をとります。
  差が大きい = 境界線 だけを残します。
  製造業の傷検出などで使われる基本技術です。」

EMBOSS（浮き彫り）:
「光源が斜めから当たったような陰影を作ります。
  立体感が生まれます。」
```

---
## 3.4 ピクセル値の統計：「明るさ」を数値で分析する

### 📢 講師メモ
「画像も数値だから統計できる」というのがこのセクションのポイントです。
平均ピクセル値 = 「明るさ」というのは直感的に理解しやすいので、
「数字1と8ではどちらが明るいか」を予想させてから確認すると盛り上がります。

In [ ]:
# PIL Image を numpy に戻して統計計算
arr = np.array(img_base)

print('=== ピクセル値の統計（数字「8」）===')
print(f'形状（縦×横）: {arr.shape}')
print(f'平均（明るさ）: {arr.mean():.1f}')
print(f'標準偏差（コントラスト）: {arr.std():.1f}')
print(f'最大値（最も明るい箇所）: {arr.max()}')
print(f'最小値（最も暗い箇所）: {arr.min()}')
print(f'描画率（0より大きいピクセルの割合）: {(arr > 0).mean()*100:.1f}%')

In [ ]:
# 数字（0〜9）ごとの「平均ピクセル値（明るさ）」を比較する
brightness_by_digit = {}
for digit in range(10):
    indices = np.where(digits.target == digit)[0]
    # 各数字の全画像の平均ピクセル値
    mean_brightness = digits.images[indices].mean()
    brightness_by_digit[digit] = mean_brightness

# 棒グラフで表示
fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(brightness_by_digit.keys(), brightness_by_digit.values(),
              color='steelblue', edgecolor='white')

# 値をバー上に表示
for bar, val in zip(bars, brightness_by_digit.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.03,
            f'{val:.2f}', ha='center', va='bottom', fontsize=9)

ax.set_title('数字ごとの平均ピクセル値（明るさ）')
ax.set_xlabel('数字（ラベル）')
ax.set_ylabel('平均ピクセル値（0〜16）')
ax.set_xticks(range(10))
plt.tight_layout()
plt.show()

### 📢 グラフから読み取れること
- **数字「1」** が最も暗い（ピクセルが少ない）→ 細いストロークだから
- **数字「8」「0」** が明るい（ピクセルが多い）→ 閉じた曲線で描かれるから
- **重要な教訓**：平均ピクセル値だけでは 0 と 8 を区別できない
  → 機械学習では「どのピクセルが高いか（位置情報）」も重要

---
## 🔑 演習の解答・解説

### 📢 演習進行のポイント
- 問1・2はビジュアル的に楽しいので受講者が自然と取り組みやすい
- 問3の「なぜ変化するか」の考察が重要。コードを実行した後に必ず「なぜ？」を問いかける
- 問4の2値化は NumPy の基本操作の練習としても有効

### 問1：好きな数字を選んで、フィルタ処理の前後を並べて表示する

In [ ]:
# ✅ 問1：解答例（数字「3」を選んだ場合）
target_digit = 3

# 最初の「3」のインデックスを取得
idx = np.where(digits.target == target_digit)[0][0]
img = prepare_digit_image(idx)

# フィルタ処理
img_contour = img.filter(ImageFilter.CONTOUR)
img_blur    = img.filter(ImageFilter.BLUR)
img_sharp   = img.filter(ImageFilter.SHARPEN)

# 並べて表示
fig, axes = plt.subplots(1, 4, figsize=(14, 4))
images = [img, img_contour, img_blur, img_sharp]
titles = ['元画像', 'CONTOUR（輪郭）', 'BLUR（ぼかし）', 'SHARPEN（シャープ）']

for ax, image, title in zip(axes, images, titles):
    ax.imshow(image, cmap='gray_r')
    ax.set_title(f'{title}\n数字: {target_digit}', fontsize=11)
    ax.axis('off')

plt.tight_layout()
plt.show()

**📢 解説のポイント：**
- CONTOUR フィルタで数字の「輪郭だけ」が残ることを確認
- これが製造業での「傷検出」「輪郭検出」の基礎技術
- 「輪郭を数値化すれば、傷の有無・形を自動判定できる」というイメージへ繋げる

### 問2：同じ数字の複数枚を並べて表示する（手書きのばらつきを体感）

In [ ]:
# ✅ 問2：解答（数字「3」の最初の8枚を表示）
target_digit = 3
indices = np.where(digits.target == target_digit)[0][:8]  # 最初の8枚

fig, axes = plt.subplots(2, 4, figsize=(12, 6))

for i, (ax, idx) in enumerate(zip(axes.flat, indices)):
    img = prepare_digit_image(idx)
    ax.imshow(img, cmap='gray_r')
    ax.set_title(f'インデックス: {idx}', fontsize=9)
    ax.axis('off')

plt.suptitle(f'同じ数字「{target_digit}」でも手書きには個人差がある', fontsize=13)
plt.tight_layout()
plt.show()

**📢 解説のポイント：**
- 同じ「3」でも書き方が全然違う！
- 人間には全部「3」に見えるが、AI（機械学習）にとっては「数値の配列が異なるデータ」
- これが画像認識が難しい理由 = 「どんなバリエーションでも同じ数字と判定する」学習が必要

> 「1797枚の学習データを見ることで、AIはこのばらつきをカバーするパターンを学習します。
> 人間の脳と同じように「この形のパターンが3だ」と学習するイメージです。」

### 問3：フィルタ処理の前後でピクセル値の統計がどう変わるか確認する

In [ ]:
# ✅ 問3：解答
idx_3 = np.where(digits.target == 3)[0][0]
img_base_3 = prepare_digit_image(idx_3)
img_blur_3    = img_base_3.filter(ImageFilter.BLUR)
img_contour_3 = img_base_3.filter(ImageFilter.CONTOUR)

results = {}
for label, img in [('元画像', img_base_3), ('BLUR後', img_blur_3), ('CONTOUR後', img_contour_3)]:
    arr = np.array(img)
    results[label] = {
        '平均（明るさ）': arr.mean(),
        '標準偏差（コントラスト）': arr.std(),
        '最大値': arr.max(),
        '描画率(%)': (arr > 0).mean() * 100
    }

import pandas as pd
result_df = pd.DataFrame(results).T.round(1)
print('=== フィルタ処理前後のピクセル統計 ===')
print(result_df)

**📢 解説：なぜ統計値が変化するか**

```
BLUR（ぼかし）後：
- 平均値はほぼ変わらない（全体の明るさは保たれる）
- 標準偏差が小さくなる（明暗の差がなめらかになるから）

CONTOUR（輪郭）後：
- 平均値が上がることが多い（輪郭線が白く残るため）
  または下がる（背景が白になり輪郭だけ暗くなる場合）
- 標準偏差が小さくなる（ほぼ均一な背景に輪郭線だけ）
```

> 「このように統計値でフィルタの効果を数値化できます。
> 実務では「フィルタ前後のピクセル統計の変化」を特徴量として機械学習に使うこともあります。」

### 問4（発展）：明るさによる2値化（閾値処理）

In [ ]:
# ✅ 問4：解答
# np.where(条件, Trueの値, Falseの値) を使った2値化

idx_0 = np.where(digits.target == 0)[0][0]
img_base_0 = prepare_digit_image(idx_0)
arr_0 = np.array(img_base_0)

threshold = 128  # 閾値：128以上なら白（255）、未満なら黒（0）

# 2値化
arr_binary = np.where(arr_0 >= threshold, 255, 0).astype(np.uint8)
img_binary = Image.fromarray(arr_binary, mode='L')

# 並べて表示
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(img_base_0, cmap='gray_r')
axes[0].set_title('元画像')
axes[0].axis('off')

# ピクセル値のヒストグラム（閾値を示す）
axes[1].hist(arr_0.flatten(), bins=30, color='steelblue', edgecolor='white')
axes[1].axvline(x=threshold, color='red', linewidth=2, label=f'閾値: {threshold}')
axes[1].set_title('ピクセル値のヒストグラム')
axes[1].set_xlabel('ピクセル値（0〜255）')
axes[1].set_ylabel('件数')
axes[1].legend()

axes[2].imshow(img_binary, cmap='gray_r')
axes[2].set_title(f'2値化後（閾値={threshold}）')
axes[2].axis('off')

plt.suptitle('2値化処理（Binarization）', fontsize=13)
plt.tight_layout()
plt.show()

print(f'2値化後のユニークな値: {np.unique(arr_binary)}')  # [0, 255] のはず

**📢 解説：2値化の実務での用途**
- QRコード・バーコードの読み取り（白黒2値で処理）
- 医療画像：腫瘍の領域を白、それ以外を黒に変換（セグメンテーション）
- 文書の文字認識（OCR）：文字（黒）と背景（白）を分離

> 「たった1行のコード `np.where(arr >= 128, 255, 0)` で画像を2値化できます。
> これが実際の画像処理の基礎技術と同じ考え方です。」

---
## ✅ チャプターのまとめ（講師用コメント付き）

| 操作 | コード | 覚えるポイント |
|------|--------|---------------|
| numpy → PIL | `Image.fromarray(arr, mode='L')` | `mode='L'` はグレースケール |
| PIL → numpy | `np.array(img)` | 統計・計算は numpy |
| リサイズ | `img.resize((幅, 高さ), Image.NEAREST)` | (幅, 高さ) の順に注意 |
| フィルタ | `img.filter(ImageFilter.BLUR)` | 種類は BLUR / SHARPEN / CONTOUR 等 |
| 2値化 | `np.where(arr >= 閾値, 255, 0)` | 閾値の選び方が重要 |

### 📢 Ch.4 への橋渡し
> 「午前中に数値データと画像データの分析を体験しました。
> 午後はいよいよ「機械学習モデル」を作ります。
> Ch.1・2 で発見した「品種ごとの特徴の違い」を、コンピュータに自動的に学ばせます。」